In [1]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus
from sqlalchemy.orm import sessionmaker

params = quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=AI;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    pool_pre_ping=True
)


SessionLocal = sessionmaker(
    bind=engine,
    autoflush=False,
    autocommit=False
)

In [2]:
from langchain_openai.chat_models import ChatOpenAI 
llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="not-needed",
    model="google/gemma-3-4b",
    temperature=0,
    max_completion_tokens=250
)

c:\Users\babaei.nima\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import pyodbc
from sqlalchemy import text

print(pyodbc.drivers())
with engine.connect() as conn:
    result = conn.execute(text("SELECT @@VERSION"))
    print(result.scalar())

In [3]:
db = SessionLocal()
session_id = "s1"
question = "how can I learn about it?"

In [ ]:
from chat_history import (
    save_message,
    load_summary,
    load_recent_messages,
    rewrite_question,
    summarize_conversation,
    save_summary,
    get_db
)
from LMStudio_Embeddings import get_embedding
from qdrant_store import search
from PromptBuilder import build_rag_prompt
from LMStudio_LLM import generate

In [ ]:
save_message(db= db, 
             session_id=session_id, 
             role="user", 
             content=question)
db.commit()
db.close()

In [5]:
summary = load_summary(db= db, 
             session_id=session_id)
print(summary)

Production planning encompasses developing a strategy for manufacturing a company’s products and services, outlining production processes, anticipated demand, and necessary requirements. Recent conversations have focused on understanding the scope of this process. Initially, users sought a general definition of production planning – it's about strategically managing how goods are made, considering both expected demand and production needs. 

More recently, users have inquired about the *future* of production planning. This indicates an interest in emerging trends and potential developments within the field.  The conversation highlights a desire to understand not just the current state but also how production planning is evolving to meet future challenges and opportunities. Further exploration into specific aspects like digitalization, sustainability, or demand forecasting would likely be beneficial.


In [6]:
history = load_recent_messages(db= db, 
             session_id=session_id)
print(history)

user: tell me about production planning
system: Production planning is the process of developing a strategy for the production of a company's products and services. It describes how goods will be manufactured, the expected demand for those goods, and any production requirements 
user: what is the future of Production Planning?
system: Production planning shifts to autonomous, real-time systems. AI and digital twins enable self-optimizing schedules. Planners become strategists, not data chasers. Integration with suppliers and logistics is automatic. The future is zero-touch for routine work, requiring trusted data and culture change by 2030.
user: how can I learn about it?


In [7]:
Modified_question = rewrite_question(question=question, history=history, summary=summary)
print(Modified_question)

Given that production planning shifts to autonomous, real-time systems driven by AI and digital twins, where planners become strategists and integration with suppliers and logistics is automatic, what resources or learning paths are available to understand this evolving field and the necessary cultural changes for its successful implementation by 2030?


In [8]:
embedded_question = get_embedding(text=Modified_question)
print(embedded_question)

[0.018967527896165848, 0.06929212808609009, -0.041160669177770615, -0.010317967273294926, -0.028721382841467857, -0.023830663412809372, -0.04302467033267021, -0.017229055985808372, 0.030748194083571434, 0.042864881455898285, -0.03798404708504677, 0.021234534680843353, -0.01167402882128954, 0.00377322337590158, 0.04138666391372681, 0.013051408343017101, -0.012273079715669155, 0.03257559612393379, -0.004569764249026775, 0.016855323687195778, 0.034635525196790695, 0.009487348608672619, -0.03455885499715805, -0.049151711165905, -0.03747696802020073, 0.04168383404612541, -0.021847687661647797, 0.01227620244026184, 0.07346835732460022, 0.04659454897046089, -0.001877772156149149, -0.025733161717653275, 0.00508838938549161, -0.034503135830163956, -0.019039010629057884, 0.022771738469600677, 0.011697635985910892, -0.00984897930175066, -0.04394741728901863, -0.060947876423597336, 0.03465443477034569, 0.002886426169425249, 0.014580115675926208, -0.07125339657068253, -0.037206921726465225, 0.02030

In [9]:
retrieved = search(query_vector=embedded_question)
print(retrieved)

points=[ScoredPoint(id=9, version=1, score=0.07655068, payload={'filename': 'CE1_EN.docx', 'chunk_index': 8, 'text': '[CE 1.4]\xa0Nature of Particular Work Area\nThe environment, in which I was engaged in design and optimization, was interactive and I collaborated with a group of people with electrical engineering expertise as well as technical technicians in the field of working with various production devices. Also, the whole process of purchasing the primary items to assembly and selling the products was done in especially in the current space of the factory'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=1, version=1, score=0.06506487, payload={'filename': 'CE1_EN.docx', 'chunk_index': 0, 'text': 'Career Episode 1\nProductivity development project of the fluorescent frame of household and office consumes\nIntroduction\n[CE 1.1] This project started at the beginning of 2015 in the company … located in Iran, Tehran, Eshtehard industrial park and It lasted for 6 mont

In [10]:
contexts = []
for h in retrieved.points:
    payload = h.payload or {}
    contexts.append({
        "score": float(h.score),
        "text": payload.get("text", ""),
        "filename": payload.get("filename", ""),
        "chunk_index": payload.get("chunk_index", None),
    })
print(contexts)

[{'score': 0.07655068, 'text': '[CE 1.4]\xa0Nature of Particular Work Area\nThe environment, in which I was engaged in design and optimization, was interactive and I collaborated with a group of people with electrical engineering expertise as well as technical technicians in the field of working with various production devices. Also, the whole process of purchasing the primary items to assembly and selling the products was done in especially in the current space of the factory', 'filename': 'CE1_EN.docx', 'chunk_index': 8}, {'score': 0.06506487, 'text': 'Career Episode 1\nProductivity development project of the fluorescent frame of household and office consumes\nIntroduction\n[CE 1.1] This project started at the beginning of 2015 in the company … located in Iran, Tehran, Eshtehard industrial park and It lasted for 6 months, with the aim of improving the production processes of lighting industry products and restudying the design of work flows and optimizing production processes and inc

In [11]:
rag = build_rag_prompt(question=Modified_question, contexts=contexts)
print(rag)


You are a helpful assistant.
Answer the question using ONLY the context below.

RULES:
    1. Use only the document context.
    2. Never invent information.
    3. If information is missing, say so.
    4. Quote relevant passages when useful.
    5. Cite document names.
    6. Keep answers clear and professional.

If the answer cannot be found in the provided context:
    1. Say the information is unavailable.
    2. Explain what information would be needed.
    3. Do not guess.

Context:
[1] (CE1_EN.docx chunk 8)
[CE 1.4] Nature of Particular Work Area
The environment, in which I was engaged in design and optimization, was interactive and I collaborated with a group of people with electrical engineering expertise as well as technical technicians in the field of working with various production devices. Also, the whole process of purchasing the primary items to assembly and selling the products was done in especially in the current space of the factory

[2] (CE1_EN.docx chunk 0)
Caree

In [ ]:
answer = generate(prompt=rag)
print(answer)

In [14]:
save_message(db= db, 
             session_id=session_id, 
             role="system", 
             content=answer)
db.commit()
db.close()

In [15]:
history2 = load_recent_messages(db= db, 
             session_id=session_id)
print(history2)

user: tell me about production planning
system: Production planning is the process of developing a strategy for the production of a company's products and services. It describes how goods will be manufactured, the expected demand for those goods, and any production requirements 
user: what is the future of Production Planning?
system: Production planning shifts to autonomous, real-time systems. AI and digital twins enable self-optimizing schedules. Planners become strategists, not data chasers. Integration with suppliers and logistics is automatic. The future is zero-touch for routine work, requiring trusted data and culture change by 2030.
user: how can I learn about it?
system: The provided context does not contain information about shifting production planning to autonomous, real-time systems driven by AI and digital twins, or the necessary cultural changes for their successful implementation by 2030. 

Specifically, it offers details regarding a fluorescent frame project from 2015 

In [16]:
new_summary = summarize_conversation(db= db, 
             session_id=session_id)
print(new_summary)

Production planning involves strategically managing a company’s product manufacturing, encompassing anticipated demand, production processes, and necessary requirements. Recent conversations have expanded this understanding to include the *future* of the field, driven by significant technological shifts. Users are increasingly interested in how production planning is evolving to meet future challenges and opportunities.

The current trajectory points towards autonomous, real-time systems leveraging AI and digital twins for self-optimizing schedules – a shift where planners transition from data analysis to strategic oversight. This “zero-touch” approach anticipates automatic integration with suppliers and logistics by 2030, contingent on a supportive organizational culture. 

Initially, the system provided a foundational definition of production planning and detailed information regarding specific factory operations (including quality control and an integrated software system) from a 20

In [17]:
save_summary(db= db, 
             session_id=session_id,
             summary_text=new_summary)
db.commit()
db.close()

In [24]:
import uuid
d = str(uuid.uuid4())

print(d)

917b531a-af7d-427d-90b8-986373c19eab
